### Pandas tutorial

#### 1. Exploring a dataframe

In [ ]:
import re
import pandas as pd
import numpy as np
from toolkits.scrollable import display_dataframe as dp

print("version:", pd.__version__)
pd.set_option("display.max_columns", None)
# pd.set_option('display.max_rows', None)

KBO = pd.read_csv(
    "./data/kbo.csv", encoding='utf-8'
)

dp(KBO)

#### 1-1. indexing



| 방법        | 사용 예시                               | 특징 |
|-------------|----------------------------------------|------|
| **df[]**      | `df['Age']`, `df[df['Age'] > 30]`      | 기본적인 열 선택 및 조건을 사용한 행 필터링. 행과 열을 동시에 선택할 수 없음. |
| **df.loc[]**  | `df.loc[0]`, `df.loc[0, 'Age']`, `df.loc[0:2]` | **라벨(이름)**을 기준으로 행과 열을 선택. 슬라이싱 시 끝 인덱스 포함. 조건식 가능. |
| **df.iloc[]** | `df.iloc[0]`, `df.iloc[0, 1]`, `df.iloc[0:2]` | **정수 인덱스**를 기준으로 행과 열을 선택. 슬라이싱 시 끝 인덱스 미포함. 조건식 불가능. |


In [ ]:
KBO[KBO['체중'] > 80].loc[KBO['선수명'].str.contains(r'강.구'), ['생년월일', '선수명', '체중']]

In [ ]:
KBO['birth'] = pd.to_datetime(KBO['생년월일'], format='%Y년 %m월 %d일')
KBO[KBO['체중'] > 80].loc[KBO['선수명'].str.contains(r'강.구'), ['birth', '생년월일', '선수명', '체중']]

Pandas에서 datetime 형식의 값을 다룰 때, 다양한 날짜 연산:

- 날짜 간의 차이를 계산: df['end_date'] - df['start_date']
- 날짜에 일수를 더하거나 빼기: pd.Timedelta(days=10)
- 현재 날짜와의 차이 계산: (today - df['start_date']).dt.days
- 날짜에서 연도, 월, 일을 추출: .dt.year, .dt.month, .dt.day

In [ ]:
KBO['start_y'] = KBO[KBO['입단년도'].notna()]['입단년도'].str.extract(r'(\d{0,2})').astype(int).apply(lambda x: 2000 + x)
KBO[KBO['입단년도'].notna()].loc[0:3, ['선수명', '생년월일', 'birth', '입단년도', 'start_y',]]

In [ ]:
from datetime import datetime
KBO['since_y'] = datetime.now().year - KBO.loc[KBO['start_y'].notna(), 'start_y'] 
KBO.loc[0:2, ['선수명', '생년월일', 'birth', '입단년도', 'start_y', 'since_y']]

In [ ]:
KBO.iloc[0:3, [0, 2, -3, -2, -1]]

In [ ]:
KBO[['연봉', '연봉(단위)']] = KBO['연봉'].str.extract(r'([0-9,.]+)\s*(\D+)')
KBO[['입단 계약금', '입단 계약금(단위)']] = KBO['입단 계약금'].str.extract(r'([0-9,.]+)\s*(\D+)')

In [ ]:
KBO.loc[0:2, ['선수명', '생년월일', 'birth', '입단년도', 'start_y', 'since_y', '연봉', '연봉(단위)', '입단 계약금', '입단 계약금(단위)']] 

#### 1-2 make group

In [ ]:
import pandas as pd
from toolkits.scrollable import display_dataframe as dp

hflight = pd.read_csv(
    './data/hflight.csv'
)
dp(hflight.head(15))

In [ ]:
ans = hflight.groupby(['Origin', 'Dest'])[['ArrDelay', 'DepDelay']].mean()
dp(ans)

In [ ]:
tmp = hflight[hflight['ArrDelay'] >= 5].groupby(['Dest'])[['ArrDelay']].agg(['count']).reset_index()
tmp.columns = ['Dest', 'Cnt']
tmp = tmp[tmp['Cnt'] > 2000]
tmp

In [ ]:
hflight.loc[mask, :].groupby(['Dest'])['Year'].count()

In [ ]:
# count 'Cancelled', 'Diverted'
mask = hflight['ArrDelay'] >= 5 

hflight.loc[mask, :].groupby(['Dest'])['Year'].count()

hflight[hflight.Dest.isin(tmp1[mask2].index)].copy()
hflight.loc[mask, :].groupby(['Dest'])['Cancelled']

In [ ]:
mask = hflight.loc[mask, :].groupby(['Dest'])['Year'].count() > 2000
hflight[hflight.Dest.isin(hflight.loc[mask, :].groupby(['Dest'])['Year'].count()[mask].index)]

In [25]:
hflight.groupby(['Dest'])['Year'].count()

Dest
ABQ    2812
AEX     724
AGS       1
AMA    1297
ANC     125
       ... 
TUL    2924
TUS    1565
TYS    1210
VPS     880
XNA    1172
Name: Year, Length: 116, dtype: int64

In [23]:
hflight.groupby(['Dest'])[['Cancelled', 'Diverted']].agg('sum')

,Cancelled,Diverted
Dest,,
ABQ,25,7
AEX,12,2
AGS,0,0
AMA,32,8
ANC,0,1
...,...,...
TUL,54,4
TUS,15,2
TYS,8,5


In [ ]:

mask2 = tmp1 >= 2000
tmp1[mask2].index

hflight2 = hflight[hflight.Dest.isin(tmp1[mask2].index)].copy()

tmp = pd.concat(
    [hflight.loc[mask, :].groupby(['Dest'])[['Year']].count(), 
    hflight.loc[mask, :].groupby(['Dest'])[['Cancelled']].sum(),
    hflight.loc[mask, :].groupby(['Dest'])[['Diverted']].sum()], axis=1
)
tmp.columns = ['Total', 'Cancelled', 'Diverted']
hflight.loc[mask, :].groupby(['Dest'])[['Year']].count()
dp(hflight.loc[mask, :].groupby(['Dest'])[['Cancelled', 'Diverted']].agg(['sum']))

In [ ]:
hflight.isnull().sum()


In [ ]:
# sum by rows
KBO.isnull().sum(axis=1)
# sum by columns
KBO.isnull().sum(axis=0)

In [ ]:
KBO.info()

In [ ]:
KBO.dtypes

In [ ]:
KBO.dtypes.value_counts()

2. Add ["back_number", "date", "deposit", "income", ...] columns

In [ ]:
pos = KBO['포지션'].unique()
pos

To create a new column based on the values of an existing column, and assign True if the value exists in that column, and False otherwise,

In [ ]:
KBO[pos] = False
for p in pos:
    KBO[p] = KBO['포지션'] == p
KBO.tail(3)

In [ ]:
# dataframe[] -> Series, dataframe[[]] -> Dataframe
KBO['화폐 단위'] = KBO['연봉'].str.extract(r'(\D+)$')
# 
KBO['입단 계약금'] = KBO['입단 계약금'].fillna(0)

crcy = KBO['화폐 단위'].unique()[~pd.isna(KBO['화폐 단위'].unique())]

for c in crcy:
    KBO[f'연봉({c})'] = KBO['연봉'].str.extract(fr'(\d+)(?={c})').fillna(0).astype(int)
    # KBO['연봉(달러)'] = KBO['연봉'].str.extract(r'(\d+)(?=달러)').fillna(0).astype(int)
    KBO[f'입단 계약금({c})'] = KBO['입단 계약금'].str.extract(fr'(\d+)(?={c})').fillna(0).astype(int)
    # KBO['입단 계약금(달러)'] = KBO['입단 계약금'].str.extract(r'(\d+)(?=달러)').fillna(0).astype(int)
KBO.tail(3)

In [ ]:
KBO['입단년도'] = KBO['입단년도'].fillna(0)
KBO['입단년도'].str.extract(r'^\d{0,2}[a-zA-Z가-힣]+$')
# KBO['입단년도'].value_counts()

In [ ]:

KBO

In [ ]:
np.isnan(crcy[2])
# np.isnan(KBO['화폐 단위'].unique().to_numpy())

In [ ]:
type(crcy[2])
crcy == np.nan
pd.isna(crcy[2])
~pd.isna(KBO['화폐 단위'].unique())
KBO['화폐 단위'].unique()[~pd.isna(KBO['화폐 단위'].unique())]

In [ ]:
KBO['birth'] = pd.to_datetime(
    KBO['생년월일'].str.replace(r'년|월|일', '-', regex=True).str.rstrip('-').str.replace(r' ', '', regex=True),
    format='%Y-%m-%d'
)
KBO.tail(3)

In [ ]:
KBO['연봉'].str.contains('달러|만원', regex=True)
KBO['화폐 단위'] = KBO['연봉'].str.extract(r'(\D+)$')
KBO['화폐 단위'].unique()

In [ ]:
KBO[KBO['입단 계약금(nan)'] != 0]

In [ ]:
KBO['등번호'].str.extract(r'')

In [ ]:
dd = '1995년 05월 07일 1996년 05월 07일'
ii = '8000만원'

tmp = re.search(r'(\d{4})년 (\d{2})월 (\d{2})일', dd)
tmp.group(3)

tmp = re.search(r'[0-9]+', ii)
tmp.group(0)

In [ ]:
# tmp = re.search(r'^\d{0,2}[a-zA-Z가-힣]+$', '12키움')
tmp = re.search(r'^[0-9]+', '12키움')
tmp = re.search(r'[a-zA-Z가-힣]*$', '0')
tmp.group(0)

In [ ]:
KBO['back_number'] = None
KBO.tail(3)

In [ ]:
KBO['체중'].plot(kind='hist')
KBO['체중'].mean()

In [ ]:
KBO.to_csv(
    './data/kbo.csv', index=False
)